In [1]:
import pandas as pd
df = pd.read_csv("Food_Delivery_Times.csv")
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


In [2]:
import pandas as pd

df = pd.read_csv("Food_Delivery_Times.csv")
print(df.shape)
print(df.isnull().sum())

(1000, 9)
Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64


In [3]:
# Check missing values
print("Missing before:\n", df.isnull().sum())

# Fill categorical columns with mode
df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])
df["Time_of_Day"] = df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0])
df["Traffic_Level"] = df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0])

# Fill numerical column with mean
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mean())

# Verify no more nulls
print("Missing after:\n", df.isnull().sum())

Missing before:
 Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64
Missing after:
 Order_ID                  0
Distance_km               0
Weather                   0
Traffic_Level             0
Time_of_Day               0
Vehicle_Type              0
Preparation_Time_min      0
Courier_Experience_yrs    0
Delivery_Time_min         0
dtype: int64


In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port="your_port_here",
    user="root",
    password="Your Password Here",
)
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS food_delivery_db")
cursor.execute("USE food_delivery_db")

cursor.execute("""
CREATE TABLE IF NOT EXISTS delivery_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    order_id INT,
    distance_km DECIMAL(6,2),
    weather VARCHAR(50),
    traffic_level VARCHAR(50),
    time_of_day VARCHAR(50),
    vehicle_type VARCHAR(50),
    preparation_time_min INT,
    courier_experience_yrs DECIMAL(4,1),
    delivery_time_min INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
cursor.close()
conn.close()
print("Table ready.")

Table ready.


In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    port="your_port_here",
    user="root",
    password="Your Password Here",
    database="food_delivery_db"
)
cursor = conn.cursor()

insert_query = """
INSERT INTO delivery_data
(order_id, distance_km, weather, traffic_level, time_of_day,
 vehicle_type, preparation_time_min, courier_experience_yrs, delivery_time_min)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cursor.execute(insert_query, (
        int(row["Order_ID"]),
        float(row["Distance_km"]),
        row["Weather"],
        row["Traffic_Level"],
        row["Time_of_Day"],
        row["Vehicle_Type"],
        int(row["Preparation_Time_min"]),
        float(row["Courier_Experience_yrs"]),
        int(row["Delivery_Time_min"])
    ))

conn.commit()
cursor.close()
conn.close()
print(f"{len(df)} rows inserted into MySQL.")

1000 rows inserted into MySQL.


In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    port="your_port_here",
    user="root",
    password="Your Password Here",
    database="food_delivery_db"
)
cursor = conn.cursor()
cursor.execute("SELECT * FROM delivery_data LIMIT 5")
for row in cursor.fetchall():
    print(row)
cursor.close()
conn.close()

(1, 522, Decimal('7.93'), 'Windy', 'Low', 'Afternoon', 'Scooter', 12, Decimal('1.0'), 43, datetime.datetime(2026, 10, 4, 22, 16, 55))
(2, 738, Decimal('16.42'), 'Clear', 'Medium', 'Evening', 'Bike', 20, Decimal('2.0'), 84, datetime.datetime(2026, 10, 4, 22, 16, 55))
(3, 741, Decimal('9.52'), 'Foggy', 'Low', 'Night', 'Scooter', 28, Decimal('1.0'), 59, datetime.datetime(2026, 10, 4, 22, 16, 55))
(4, 661, Decimal('7.44'), 'Rainy', 'Medium', 'Afternoon', 'Scooter', 5, Decimal('1.0'), 37, datetime.datetime(2026, 10, 4, 22, 16, 55))
(5, 412, Decimal('19.03'), 'Clear', 'Low', 'Morning', 'Bike', 16, Decimal('5.0'), 68, datetime.datetime(2026, 10, 4, 22, 16, 55))
